# clue_text

> The clue board in words (Phase 11): rules with a worked example, observations with the open subtasks, own and shared clues and the open puzzle, parsing of open, submit, share and wait (anything unusable becomes a counted wait), and posting shared clues to the channel.

In [ ]:
#| default_exp clue_text

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re

from explRL.clue_agents import ClueBoard, ClueLead, ClueMessages, candidates, known

_STR = {'type': 'string'}
CLUE_SCHEMA = {'type': 'object', 'additionalProperties': False, 'required': ['action', 'target', 'answer', 'clue', 'notes'],
               'properties': {'action': {'type': 'string', 'enum': ['open', 'submit', 'share', 'wait']}, 'target': _STR,
                              'answer': {'type': 'integer'}, 'clue': _STR, 'notes': _STR}}
CLUE_RE = re.compile(r'x\s*mod\s*(\d+)\s*=\s*(-?\d+)', re.I)


def _fmt(p, r):
    return f'x mod {p} = {r}'


class TextClueEnv:
    "The clue board in words: open subtasks, own and shared clues, the open puzzle; parsed actions; shares posted."
    task, timeUnit, SCHEMA, unit = 'clue', 'turn', CLUE_SCHEMA, 'points'

    def __init__(self, board):
        self.world, self.shown, self._heard, self._known = board, {}, {}, {}

    def rules(self):
        b = self.world
        return (f"You are one of a team of {b.nAgents} solvers on a shared board of subtasks. The board always shows "
                f"{2 * b.nAgents + 2} open subtasks, named T followed by a number, worth {', '.join(f'{v:g}' for v in sorted(set(b.values)))} "
                f"points each and open for {b.life:g} turns after they appear; a solved or expired subtask is replaced at once.\n\n"
                f"Each subtask hides a whole number x with 0 <= x < {b.R}. The answer is fixed by {b.m} clues of the form "
                "'x mod p = r' (the remainder of x divided by p is r), one for each of the subtask's primes. All of a "
                "subtask's clues together give exactly one answer; with one clue missing there are always several "
                "possible answers. The clues are dealt to different solvers: usually nobody holds all clues of a "
                "subtask, so the team only solves it if clues reach whoever works on it.\n\n"
                "Worked example: 0 <= x < 105 with x mod 3 = 2, x mod 5 = 3 and x mod 7 = 2. List the numbers below 105 "
                "with remainder 2 when divided by 7: 2, 9, 16, 23, 30, ... Test them against the other clues: 23 mod 5 = "
                "3 and 23 mod 3 = 2, so x = 23. Without the clue x mod 3 = 2, both 23 and 58 would fit.\n\n"
                "Turns: each of your turns is one action and takes one turn. 'open' a subtask (target = its name) to work "
                "on it from your next turn on. 'submit' an answer (an integer in answer) to the subtask you have open; "
                "you may answer again after a wrong answer while it is open. 'share' one clue you know (your own, or one "
                "you have seen) for an open subtask: target = the subtask, clue = the clue text, for example "
                "'x mod 5 = 3'; sharing takes your turn. 'wait' does nothing.\n\n"
                f"Scoring: only the first correct answer to a subtask scores its value; a correct answer after it is "
                f"solved or expired scores nothing; every wrong answer costs {b.penalty:g} points. Depending on the "
                "team's set-up, a shared clue goes to a board every teammate sees, to two random teammates as a message, "
                "or to a coordinator who passes it to whoever has that subtask open and suggests subtasks; or nowhere. "
                "A clue shared once is known to everyone who saw it, so sharing it again only helps teammates who did "
                "not see it.\n\n"
                "What you see each turn: the turn, what your last action earned, the open subtasks with their values, the "
                "turns they have left and how many of their clues you know, your own clues, the clues your team's set-up "
                "has shown you, and, for your open subtask, every clue you know and the answer range. Below that come "
                "your own past turns (or your latest notes).\n\n"
                "Reply with JSON only: action is 'open', 'submit', 'share' or 'wait'; target is the subtask name for "
                "'open' and 'share' (empty otherwise); answer is your integer for 'submit' (0 otherwise); clue is the "
                "clue text for 'share' (empty otherwise); notes is a short note to yourself, shown back to you later.")

    def observe(self, i, t, channel=None):
        b = self.world
        b.advance(t)
        o, kn = b.observe(i), known(i, b, channel)
        self._known[i] = kn
        lines = [f"Turn {t:g}. Your last action earned {o['lastReward']:.2f}.",
                 'Open subtasks (value, turns left, clues you know): ' +
                 ', '.join(f'T{k} {v:g} ({e - t:g} left, {len(kn[k])}/{b.m})' for k, v, e in o['open'])]
        own = [f'T{k}: {_fmt(*b.tasks[k].task.clue(j))}' for k in b.open for j in sorted(b.held(i, k))]
        lines.append('Your own clues: ' + ('; '.join(own) or 'none'))
        shown = set()
        if isinstance(channel, ClueBoard):
            view = channel.view(i, t, b)
            shown = set(view)
            lines.append('Clues shared on the board: ' + ('; '.join(
                f'T{k}: {_fmt(p, r)} (agent {a}, {t - ts:g} turns ago)' for k, xs in view.items() for _, p, r, a, ts in xs) or 'none'))
        elif isinstance(channel, ClueMessages):
            msgs = channel.take(i)
            shown = {k for _, k, _, _ in msgs}
            self._heard[i] = [f'T{k}: {_fmt(c[1], c[2])}' for _, k, c, _ in msgs]
            lines.append('Messages: ' + ('; '.join(f'agent {a} says T{k}: {_fmt(c[1], c[2])}' for a, k, c, _ in msgs) or 'none'))
        elif isinstance(channel, ClueLead):
            routed = [x for x in channel.routed[i] if x[1] in b.open]
            j = channel.suggest(i, b)
            shown = {x[1] for x in routed} | {j}
            lines.append('Clues the coordinator passed to you: ' + ('; '.join(f'T{k}: {_fmt(c[1], c[2])} (from agent {a})'
                                                                              for a, k, c, _ in routed) or 'none'))
            lines.append(f'The coordinator suggests: T{j}.')
        k = o['opened']
        if k is None:
            lines.append('You have no subtask open.')
        else:
            state = 'open' if k in b.open else 'closed (solved or expired)'
            cl = '; '.join(_fmt(b.tasks[k].task.primes[j], r) for j, r in sorted(kn.get(k, {}).items())) or 'none'
            lines.append(f'Your open subtask: T{k}, {state}. Find x with 0 <= x < {b.R}. Clues you know: {cl}.')
        self.shown[i] = shown
        return '\n'.join(lines)

    def _task(self, s):
        s = str(s).strip().upper()
        return int(s[1:]) if s[:1] == 'T' and s[1:].isdigit() else None

    def parse(self, parsed, i):
        "-> (action, report (k, clue) for shares or None, valid); anything unusable is 'wait', invalid."
        b, wait = self.world, ('wait', None)
        if not isinstance(parsed, dict):
            return wait, None, False
        a, k = parsed.get('action'), self._task(parsed.get('target', ''))
        if a == 'wait':
            return wait, None, True
        if a == 'open':
            return (('open', k), None, True) if k in b.open else (wait, None, False)
        if a == 'share' and k in b.open and (m := CLUE_RE.search(str(parsed.get('clue', '')))):
            p, r = int(m.group(1)), int(m.group(2))
            j = next((j for j, q in enumerate(b.tasks[k].task.primes) if q == p), None)
            if j is not None and self._known.get(i, {}).get(k, {}).get(j) == r:
                return ('share', (k, j, r)), (k, (j, p, r)), True
            return wait, None, False
        ans, ko = parsed.get('answer'), b.opened[i]
        if a == 'submit' and ko is not None and isinstance(ans, int) and not isinstance(ans, bool):
            if ko in b.open:
                b.noteSubmit(i, len(candidates(b, ko, self._known.get(i, {}).get(ko, {}))))
            return ('submit', ans), None, True
        return wait, None, False

    def post(self, i, report, t, channel, info):
        if report is None or channel is None:
            return False
        k, clue = report
        if isinstance(channel, ClueLead):
            channel.post(i, k, clue, t, self.world)
        else:
            channel.post(i, k, clue, t)
        return True

    def explored(self, i, action, info):
        "Opening a subtask no shared clue or suggestion pointed to this turn."
        return info['kind'] == 'travel' and action[1] not in self.shown.get(i, ())

    def summary(self, i, t, action, reward, report, info):
        res, k = info['result'], info['task']
        verb = dict(opened=f'opened T{k}', solved=f'solved T{k}', wrong=f'answered {action[1]} to T{k}: wrong',
                    late=f'answered T{k} too late', shared=f'shared T{k}: {_fmt(*report[1][1:]) if report else ""}',
                    wait='waited', invalid='made an invalid move')[res]
        s = f'turn {t:g}: {verb}, earned {reward:.2f}'
        if self._heard.get(i):
            s += '; messages: ' + ', '.join(self._heard[i])
        return s + '.'

## Tests

In [ ]:
import json
import numpy as np
from explRL.clue_text import *
from explRL.clue_agents import ClueBoard, ClueLead, ClueMessages
from explRL.clueboard import ClueBoardEnv
from explRL.llm_agent import LLMMember, runTurns
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient
from explRL.multi import spawnRngs

b = ClueBoardEnv(0, 4, m=3, r=1, R=105, life=100)
env = TextClueEnv(b)
assert len(env.rules()) / 5 > 512, len(env.rules())
k = b.open[0]; task = b.tasks[k].task
A = min(b.tasks[k].holders[0]); O = (A + 1) % 4          # A holds clue 0 of k; O will open k
own = b.held(A, k)
cb = ClueBoard()
obs = env.observe(A, 0.0, cb)
assert all(f'T{x} ' in obs for x in b.open) and 'Your own clues:' in obs and 'no subtask open' in obs
assert all(f'T{k}: x mod {task.primes[j]} = {r}' in obs for j, r in own.items())
P = lambda **kw: dict(dict(action='wait', target='', answer=0, clue='', notes=''), **kw)
# parsing: open, share (a known clue only), submit, wait; unusable replies are invalid waits
assert env.parse(P(action='open', target=f't{k}'), A) == (('open', k), None, True)
j, r = next(iter(own.items())); p = task.primes[j]
assert env.parse(P(action='share', target=f'T{k}', clue=f'x mod {p} = {r}'), A) == (('share', (k, j, r)), (k, (j, p, r)), True)
assert env.parse(P(action='share', target=f'T{k}', clue=f'x mod {p} = {(r + 1) % p}'), A)[2] is False   # not a clue it knows
unknown = [jj for jj in range(3) if jj not in own]
if unknown:
    q = task.primes[unknown[0]]
    assert env.parse(P(action='share', target=f'T{k}', clue=f'x mod {q} = {task.clue(unknown[0])[1]}'), A)[2] is False
assert env.parse(P(action='submit', answer=3), A)[2] is False and env.parse(None, A)[2] is False
# a share posted to the board becomes known to teammates; the open subtask lists every known clue and the range
b.apply(A, ('share', (k, j, r)), 0.0); env.post(A, (k, (j, p, r)), 0.0, cb, {})
b.apply(O, ('open', k), 0.0)
o2 = env.observe(O, 1.0, cb)
assert f'T{k}: x mod {p} = {r} (agent {A}' in o2 and f'Your open subtask: T{k}, open. Find x with 0 <= x < 105.' in o2
assert f'x mod {p} = {r}' in o2.split('Clues you know: ')[1]
# submitting reports the candidate count to the board
env.parse(P(action='submit', answer=task.x), O); assert b._noted[O] >= 1

# a fake-client team shares through each channel (messages to two teammates; the coordinator passes clues on)
def script(req):
    o = req['messages'][0]['content'][-1]['text']
    import re
    m = re.search(r'Your own clues: T(\d+): (x mod \d+ = \d+)', o)
    if m:
        return json.dumps(P(action='share', target=f'T{m.group(1)}', clue=m.group(2)))
    return json.dumps(P(action='wait'))
for name, ch in (('board', ClueBoard()), ('messages', ClueMessages(4, 2, spawnRngs(1, 4))), ('lead', ClueLead(4))):
    bd = ClueBoardEnv(5, 4, m=3, r=1, R=105)
    e = TextClueEnv(bd)
    ms = [LLMMember(LLMClient('claude-haiku-5-5', client=FakeClient(script)), e, ch) for _ in range(4)]
    runTurns(ms, bd, 3)
    assert bd.counts['shares'] > 0 and all(x['valid'] for m in ms for x in m.log), name
    if name == 'messages':
        assert any('agent ' in x['obs'] and 'says T' in x['obs'] for m in ms for x in m.log)
    if name == 'lead':
        assert all('The coordinator suggests: T' in x['obs'] for m in ms for x in m.log)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()